<a href="https://colab.research.google.com/github/IbrahimMahidpur/ANN_Model/blob/main/HyperParameterTuning(nodesInLayers).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
!pip uninstall scikeras -y
!pip uninstall scikit-learn -y
!pip uninstall numpy -y
!pip install scikit-learn==1.2.2
!pip install numpy==1.23.5
!pip install scikeras==0.8.1

In [5]:
dataset = pd.read_csv('/content/Churn_Modelling.csv')
X=dataset.iloc[:,3:13]
y=dataset.iloc[:,13]

In [6]:
geography=pd.get_dummies(X["Geography"],drop_first=True)
gender=pd.get_dummies(X['Gender'],drop_first=True)

In [7]:
X=pd.concat([X,geography,gender],axis=1)
X=X.drop(['Geography','Gender'],axis=1)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=0)


In [ ]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [ ]:
#performing hyper-paramter tuning


In [ ]:
import keras
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
from keras.models import Sequential
from keras.layers import Dense,Activation,Embedding,Flatten,LeakyReLU,BatchNormalization,Dropout
from keras.activations import relu

In [ ]:
from keras.models import Sequential
def create_model(layers,activation):
  model=Sequential()
  for i, nodes in enumerate(layers):
    if i==0:
      model.add(Dense(nodes,input_dim=X_train.shape[1]))
      model.add(Activation(activation))
      model.add(Dropout(0.3))
    else:
      model.add(Dense(nodes))
      model.add(Activation(activation))
      model.add(Dropout(0.3))

  model.add(Dense(units=1,kernel_initializer='glorot_uniform',activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
model=KerasClassifier(build_fn=create_model, verbose =0)


In [ ]:
layers = [[20],[40,20],[45,30,15]]
activations = ['sigmoid','relu']
param_grid = dict(layers=layers,activation=activations, batch_size=[128,256],epochs=30)
grid=GridSearchCV(estimator=model,param_grid=param_grid,cv=5)

In [ ]:
grid_result=grid.fit(X_train,y_train)

In [ ]:
print(grid_result.best_scores_,grid_result.best_params_)

In [ ]:
pred_y = grid.predict(X_test)
y_pred=( pred_y > 0.5 )

In [ ]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)

In [ ]:
score=accuracy_score(y_test,y_pred)